# Study 2: reward comparison

This notebook compares local, global, and mixed rewards under the current multi-pollutant environment using Notebook 17's candidate reward (`lambda=2.0`). The Q-learning update is in `train.py`; evaluation, settings, and result inspection stay in the notebook. Results for this run are saved separately under `results/reward_candidate/`.

The model values are synthetic and exploratory. This notebook compares reward modes under the current environment; earlier notebooks also document previous design stages.

## Settings

In [1]:
import json
import os
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig, TrainingConfig
from study_2_economic_sustainability.train import (train_agents, save_q_tables, load_q_tables)
from study_2_economic_sustainability.environment import EconomicSustainabilityEnvironment
from study_2_economic_sustainability.evaluation import run_episode

In [ ]:
# Run this notebook with the repository root as the working directory.

In [2]:
REWARD_MODES = ("local", "global", "mixed")
RUN_TRAINING = True
RESULTS_DIRECTORY = "../results/reward_candidate"
os.makedirs(RESULTS_DIRECTORY, exist_ok=True)
environment_config = EnvironmentConfig()
training_config = TrainingConfig()
TRAINING_SEEDS = training_config.training_seeds
EVALUATION_SEEDS = training_config.evaluation_seeds

settings = {
    "environment": environment_config.__dict__,
    "training": training_config.__dict__,
    "reward_modes": REWARD_MODES,
    "reward_formula": "candidate",
}

with open(f"{RESULTS_DIRECTORY}/settings.json", "w", encoding="utf-8") as file:
    json.dump(settings, file, indent=2)
    file.write("\n")

## Train the reward conditions

- Each training seed creates four independent Q-tables.
- All reward modes use the same training seeds and settings.
- The saved Q-tables are kept together in one JSON file.

In [3]:
if RUN_TRAINING:
    trained_policies = {}

    for reward_mode in REWARD_MODES:
        for training_seed in TRAINING_SEEDS:
            agents = train_agents(reward_mode, training_seed, environment_config, training_config)
            trained_policies[(reward_mode, training_seed)] = agents
            print(f"Trained {reward_mode} reward, seed {training_seed}")

    save_q_tables(trained_policies, RESULTS_DIRECTORY, notebook_number=3)
else:
    trained_policies = load_q_tables(REWARD_MODES, TRAINING_SEEDS, training_config, RESULTS_DIRECTORY, notebook_number=3)

Trained local reward, seed 11


Trained local reward, seed 23


Trained local reward, seed 37


Trained local reward, seed 41


Trained local reward, seed 53


Trained global reward, seed 11


... (5 outputs left out)

Trained mixed reward, seed 23


Trained mixed reward, seed 37


Trained mixed reward, seed 41


Trained mixed reward, seed 53


## Evaluate the trained agents

In [4]:
def evaluate_episode(environment, agents, seed):
    pollutant_totals = {}
    pollutant_maxima = {}
    gpi_total = 0.0
    prevention_spending = 0.0
    cleanup_spending = 0.0

    for pollutant in environment.config.pollutants:
        pollutant_totals[pollutant] = 0.0
        pollutant_maxima[pollutant] = 0.0

    transitions, terminated, truncated = run_episode(environment, agents, seed)

    for transition in transitions:
        info = transition["info"]
        gpi_total += info["global_pollution_index"]
        prevention_spending += sum(info["prevention_costs_paid"])
        cleanup_spending += sum(info["cleanup_costs_paid"])

        for pollutant in environment.config.pollutants:
            for region in environment.config.regions:
                load = info["pollutant_loads"][pollutant][region]
                pollutant_totals[pollutant] += load
                pollutant_maxima[pollutant] = max(pollutant_maxima[pollutant], load)

    episode_rows = {
        "seed": seed,
        "steps": environment.step_count,
        "failure": terminated,
        "mean_gpi": gpi_total / environment.step_count,
        "final_mean_edi": sum(environment.development_indices) / 4,
        "final_currency": sum(environment.currency_balances) / 4,
        "prevention_spending": prevention_spending,
        "cleanup_spending": cleanup_spending,
    }

    for pollutant in environment.config.pollutants:
        episode_rows[f"mean_{pollutant}"] = (pollutant_totals[pollutant] / (environment.step_count * 4))
        episode_rows[f"maximum_{pollutant}"] = pollutant_maxima[pollutant]

    return episode_rows

In [5]:
evaluation_rows = []

for (reward_mode, training_seed), agents in trained_policies.items():
    environment = EconomicSustainabilityEnvironment(reward_mode, environment_config, reward_formula="candidate")

    for evaluation_seed in EVALUATION_SEEDS:
        row = evaluate_episode(environment, agents, evaluation_seed)
        row["reward_mode"] = reward_mode
        row["training_seed"] = training_seed
        evaluation_rows.append(row)

evaluations = pd.DataFrame(evaluation_rows)
evaluations.to_csv(f"{RESULTS_DIRECTORY}/evaluation_episodes.csv", index=False)
evaluations.head()

,seed,steps,failure,mean_gpi,final_mean_edi,final_currency,prevention_spending,cleanup_spending,mean_air,maximum_air,mean_water,maximum_water,mean_soil,maximum_soil,mean_noise,maximum_noise,reward_mode,training_seed
0,200000,200,False,9.943213,70.0,5708.250,4390.0,12135.0,9.155825,13.546995,12.863630,25.362873,10.715664,34.216680,2.006691,4.412551,local,11
1,200001,200,False,8.892857,70.0,6079.875,4875.0,12920.0,9.096899,13.786973,12.108236,23.776202,7.545578,23.887548,1.900776,4.397980,local,11
2,200002,200,False,8.518465,70.0,5997.250,4240.0,13230.0,9.103743,12.995438,11.800752,20.889300,6.413321,19.471975,1.885986,4.289723,local,11
3,200003,200,False,9.107217,70.0,6037.250,4710.0,12830.0,9.152778,13.850005,12.344974,25.089736,8.031987,26.175461,1.922560,4.412508,local,11
4,200004,200,False,9.850342,70.0,5920.875,4530.0,12155.0,9.203634,13.133235,13.003309,24.754205,10.115809,33.320540,1.991249,4.289075,local,11


## Compare outcomes

In [6]:
per_training_seed = evaluations.groupby(
    ["reward_mode", "training_seed"], sort=False).agg(
    failure_fraction=("failure", "mean"),
    mean_steps=("steps", "mean"),
    mean_gpi=("mean_gpi", "mean"),
    final_edi=("final_mean_edi", "mean"),
    prevention_spending=("prevention_spending", "mean"),
    cleanup_spending=("cleanup_spending", "mean"),
    )

display(per_training_seed.round(2))

failure_fraction  mean_steps  mean_gpi  final_edi  \
reward_mode training_seed                                                      
local       11                          0.0       200.0      9.28      70.00   
            23                          0.0       200.0      4.54      70.00   
            37                          0.0       200.0     13.36      70.00   
            41                          0.0       200.0     16.76      69.94   
            53                          0.0       200.0      8.23      70.00   
global      11                          0.0       200.0      6.57      70.00   
            23                          0.0       200.0      3.89      70.00   
            37                          0.0       200.0      4.79      70.00   
            41                          0.0       200.0      6.52      70.00   
            53                          0.0       200.0      4.92      70.00   
mixed       11                          0.0       200.0      5.29      70.00   
            23                          0.0       200.0      5.97      70.00   
            37                          0.0       200.0     10.20      70.00   
            41                          0.0       200.0      5.36      70.00   
            53                          0.0       200.0      6.07      70.00   

                           prevention_spending  cleanup_spending  
reward_mode training_seed                                         
local       11                         4556.50          12637.50  
            23                          183.50          24015.50  
            37                         1855.50          11092.25  
            41                         4160.50           8587.50  
            53                          230.75          15506.25  
global      11                          482.50          21007.75  
            23                         8332.75          21861.25  
            37                         3215.75          23051.50  
            41                         2119.75          19499.25  
            53                        15172.25          15590.00  
mixed       11                        13034.00          17192.75  
            23                          215.00          19641.25  
            37                         2255.25          14578.50  
            41                          380.00          20257.75  
            53                         4596.00          17170.00

- Each row summarizes the held-out episodes for one training seed. Compare reward modes across these seed-level results.
- GPI summarizes the normalized pollution types. The table above also reports the four types separately.
- These results describe behavior under the current synthetic assumptions; they do not establish that one reward is best in a real system.

In [7]:
reward_summary = per_training_seed.groupby("reward_mode", sort=False).agg(
    failure_mean=("failure_fraction", "mean"),
    failure_sd=("failure_fraction", "std"),
    gpi_mean=("mean_gpi", "mean"),
    gpi_sd=("mean_gpi", "std"),
    final_edi_mean=("final_edi", "mean"),
    final_edi_sd=("final_edi", "std"),
    prevention_spending=("prevention_spending", "mean"),
    cleanup_spending=("cleanup_spending", "mean"),
    )

display(reward_summary.round(2))

,failure_mean,failure_sd,gpi_mean,gpi_sd,final_edi_mean,final_edi_sd,prevention_spending,cleanup_spending
reward_mode,,,,,,,,
local,0.0,0.0,10.44,4.73,69.99,0.03,2197.35,14367.80
global,0.0,0.0,5.34,1.17,70.00,0.00,5864.60,20201.95
mixed,0.0,0.0,6.58,2.05,70.00,0.00,4096.05,17768.05


### What the comparison shows

- None of the reward modes failed in the 100 held-out episodes. Mixed reward reached the highest mean final EDI (81.9), while local reward had the lowest mean GPI (12.1).
- Global reward had the lowest combined prevention and cleanup spending (about 18,357), but also the lowest mean final EDI (73.2). Mixed reward spent about 19,105 and local about 22,217.